# Formula 1 Data Science Project
## Module 3 — Supervised Learning: Naïve Bayes, Decision Trees, and Regression

This notebook covers:
- **Overview**: Data Science Lifecycle and Module 3 goals
- **Naïve Bayes TAB**: MN NB, Gaussian NB, Bernoulli NB
- **Decision Tree TAB**: Three trees with different root nodes
- **Regression TAB**: Logistic Regression theory + code
- **Model Comparison**: DT vs NB vs Logistic Regression — which works best for F1 prediction?

---
## Overview

The Data Science Lifecycle begins with an idea/goal/topic and utilizes data to explore, evaluate, visualize, model, gain information, and communicate results and conclusions about that topic.

During the Module 2 Project Assignment, we applied unsupervised techniques — PCA, K-Means, Hierarchical Clustering, DBSCAN, and Association Rule Mining — to F1 lap telemetry data collected via the FastF1 API. We uncovered natural structure in the data: tyre compound groupings, lap-time patterns across circuits, and frequent driver behavior associations.

In Module 3, we shift to **supervised learning**. Now that we know the labels exist (tyre compound, podium finish, fast-lap status), we can train models to *predict* those labels from lap features. This is the core of predictive analytics in motorsport — predicting pit strategy, race outcomes, and driver performance.

We will train and compare three classifiers — **Naïve Bayes**, **Decision Trees**, and **Logistic Regression** — on an F1 binary classification task: predicting whether a lap was completed on **SOFT** tyres vs. other compounds (MEDIUM/HARD). This binary framing gives a clean, interpretable target for all three model families, allowing a fair head-to-head comparison.

---
## Setup & Imports

In [3]:
# Install dependencies (run once)
!pip install fastf1 scikit-learn matplotlib seaborn pandas numpy -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [5]:
import warnings
warnings.filterwarnings('ignore')

!pip install fastf1

import fastf1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, classification_report
)

# Naive Bayes
from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB

# Decision Tree
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

# Logistic Regression
from sklearn.linear_model import LogisticRegression

fastf1.Cache.enable_cache('f1_cache')
print('All libraries loaded successfully.')

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'fastf1'

---
## Data Collection & Label Engineering

We reuse the same FastF1 lap telemetry pipeline from Module 2 (2023 season, 5 races).
The **target label** we will predict is `IsSoft` — a binary flag:
- `1` → the lap was completed on SOFT compound tyres
- `0` → MEDIUM or HARD compound

This is a meaningful real-world prediction: teams and analysts want to know from raw lap metrics alone whether a driver is on a fast, degrading SOFT tyre or a conservative compound.

In [ ]:
races_to_load = [
    (2023, 'Bahrain Grand Prix',       'R'),
    (2023, 'Saudi Arabian Grand Prix', 'R'),
    (2023, 'Australian Grand Prix',    'R'),
    (2023, 'Miami Grand Prix',         'R'),
    (2023, 'Monaco Grand Prix',        'R'),
]

all_laps = []
for year, gp, sess in races_to_load:
    try:
        session = fastf1.get_session(year, gp, sess)
        session.load()
        laps = session.laps[[
            'Driver', 'LapNumber', 'LapTime', 'Sector1Time',
            'Sector2Time', 'Sector3Time', 'SpeedI1', 'SpeedI2',
            'SpeedFL', 'SpeedST', 'Compound', 'TyreLife',
            'TrackStatus', 'IsPersonalBest'
        ]].copy()
        laps['GrandPrix'] = gp
        all_laps.append(laps)
        print(f'Loaded {gp}: {len(laps)} laps')
    except Exception as e:
        print(f'Could not load {gp}: {e}')

laps_raw = pd.concat(all_laps, ignore_index=True)
print(f'\nTotal laps: {laps_raw.shape[0]}')
laps_raw.head()

In [ ]:
def to_seconds(t):
    try:
        return t.total_seconds()
    except:
        return np.nan

time_cols = ['LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time']
for col in time_cols:
    laps_raw[col] = laps_raw[col].apply(to_seconds)

# Keep only SOFT / MEDIUM / HARD laps
laps_clean = laps_raw[laps_raw['Compound'].isin(['SOFT', 'MEDIUM', 'HARD'])].copy()

# Binary target label
laps_clean['IsSoft'] = (laps_clean['Compound'] == 'SOFT').astype(int)

# Feature columns
feature_cols = [
    'LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
    'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'TyreLife'
]

laps_clean = laps_clean.dropna(subset=feature_cols + ['IsSoft'])
laps_clean = laps_clean.reset_index(drop=True)

print(f'Clean dataset: {laps_clean.shape[0]} laps x {len(feature_cols)} features')
print(f'\nLabel distribution:')
print(laps_clean['IsSoft'].value_counts().rename({0: 'NOT SOFT (0)', 1: 'SOFT (1)'}))
laps_clean[feature_cols + ['IsSoft']].head()

---
## Train / Test Split

All supervised models share the **same** 80/20 train-test split so results are directly comparable.

The Training Set and Testing Set **must be disjoint** — if the model sees test examples during training, it memorizes answers rather than learning patterns. `train_test_split` with `random_state=42` guarantees reproducibility and zero overlap.

In [ ]:
X = laps_clean[feature_cols].values
y = laps_clean['IsSoft'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set : {X_train.shape[0]} samples  ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Testing  set : {X_test.shape[0]}  samples  ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'Overlap check: {len(set(range(len(X_train))) & set(range(len(X_train), len(X))))} shared indices (must be 0)')

# Visualize the split
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(['Dataset'], [X_train.shape[0]], color='#e63946', label=f'Train ({X_train.shape[0]})')
ax.barh(['Dataset'], [X_test.shape[0]],  left=[X_train.shape[0]], color='#457b9d', label=f'Test ({X_test.shape[0]})')
ax.set_xlabel('Number of Laps')
ax.set_title('Train / Test Split (80 / 20)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('img/train_test_split.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → img/train_test_split.png')

---
# NAIVE BAYES TAB

## (a) Overview

**Naïve Bayes (NB)** is a family of probabilistic classifiers built on Bayes' Theorem with the "naïve" assumption that all features are **conditionally independent** given the class label. Despite this strong (often violated) assumption, NB models are fast, interpretable, and surprisingly competitive — especially on text and high-dimensional data.

**Bayes' Theorem:**
$$P(y \mid X) = \frac{P(X \mid y) \cdot P(y)}{P(X)}$$

The model picks the class $y$ that maximizes $P(y \mid X)$.

### Flavors of Naïve Bayes

| Flavor | Assumes | Best For | Feature Type |
|---|---|---|---|
| **Multinomial NB** | Features are counts/frequencies | Text classification, word counts | Non-negative integers |
| **Gaussian NB** | Features follow a Normal distribution | Continuous numeric features | Real-valued |
| **Bernoulli NB** | Features are binary (0/1) | Document presence/absence | Binary |
| **Categorical NB** | Features are discrete categories | Nominal data | Integer-encoded categories |

**For this project:**
- **Gaussian NB** fits naturally — lap times and speeds are continuous real-valued measurements.
- **Bernoulli NB** works after binarizing features at their medians.
- **Multinomial NB** works after scaling features to non-negative integers.

GNB is expected to be the strongest here because the data is genuinely continuous. MN NB and Bernoulli NB lose information through discretization but are included to demonstrate the full NB family.

## (b) Data Preparation for Naïve Bayes

Each NB flavor requires a different feature representation:
- **Gaussian NB** → raw continuous values (StandardScaler for stability)
- **Bernoulli NB** → binary features (above/below median per feature)
- **Multinomial NB** → non-negative integer counts (MinMaxScaler × 100, rounded)

In [ ]:
# --- Gaussian NB: StandardScaled continuous features ---
scaler_std = StandardScaler()
X_train_gnb = scaler_std.fit_transform(X_train)
X_test_gnb  = scaler_std.transform(X_test)

# --- Bernoulli NB: binarize at training-set medians ---
train_medians = np.median(X_train, axis=0)
X_train_bnb = (X_train > train_medians).astype(int)
X_test_bnb  = (X_test  > train_medians).astype(int)

# --- Multinomial NB: scale to [0,1] then × 100 → non-negative integers ---
scaler_mm = MinMaxScaler()
X_train_mnb = np.round(scaler_mm.fit_transform(X_train) * 100).astype(int)
X_test_mnb  = np.round(scaler_mm.transform(X_test)      * 100).astype(int)

print('Feature representation shapes (all the same):')
print(f'  Gaussian NB  train: {X_train_gnb.shape}  test: {X_test_gnb.shape}')
print(f'  Bernoulli NB train: {X_train_bnb.shape}  test: {X_test_bnb.shape}')
print(f'  Multinomial NB train: {X_train_mnb.shape}  test: {X_test_mnb.shape}')

# Preview the three prepared dataframes
df_gnb = pd.DataFrame(X_train_gnb[:5], columns=feature_cols)
df_bnb = pd.DataFrame(X_train_bnb[:5], columns=feature_cols)
df_mnb = pd.DataFrame(X_train_mnb[:5], columns=feature_cols)

print('\n--- Gaussian NB features (sample, standardized) ---')
print(df_gnb.round(3).to_string())
print('\n--- Bernoulli NB features (sample, binarized) ---')
print(df_bnb.to_string())
print('\n--- Multinomial NB features (sample, integer counts) ---')
print(df_mnb.to_string())

## (c) Naïve Bayes — Code

In [ ]:
# Train all three NB models
gnb  = GaussianNB()
bnb  = BernoulliNB()
mnb  = MultinomialNB()

gnb.fit(X_train_gnb, y_train)
bnb.fit(X_train_bnb, y_train)
mnb.fit(X_train_mnb, y_train)

# Predictions
y_pred_gnb = gnb.predict(X_test_gnb)
y_pred_bnb = bnb.predict(X_test_bnb)
y_pred_mnb = mnb.predict(X_test_mnb)

# Accuracy
acc_gnb = accuracy_score(y_test, y_pred_gnb)
acc_bnb = accuracy_score(y_test, y_pred_bnb)
acc_mnb = accuracy_score(y_test, y_pred_mnb)

print(f'Gaussian NB   Accuracy: {acc_gnb:.4f} ({acc_gnb*100:.2f}%)')
print(f'Bernoulli NB  Accuracy: {acc_bnb:.4f} ({acc_bnb*100:.2f}%)')
print(f'Multinomial NB Accuracy: {acc_mnb:.4f} ({acc_mnb*100:.2f}%)')

## (d) Naïve Bayes — Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Naïve Bayes — Confusion Matrices (IsSoft Prediction)', fontsize=14, fontweight='bold')

configs = [
    (y_pred_gnb, f'Gaussian NB\nAccuracy: {acc_gnb*100:.1f}%',   axes[0]),
    (y_pred_bnb, f'Bernoulli NB\nAccuracy: {acc_bnb*100:.1f}%',  axes[1]),
    (y_pred_mnb, f'Multinomial NB\nAccuracy: {acc_mnb*100:.1f}%', axes[2]),
]

for y_pred, title, ax in configs:
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['NOT SOFT', 'SOFT'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontweight='bold')

plt.tight_layout()
plt.savefig('img/nb_confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → img/nb_confusion_matrices.png')

In [ ]:
# Accuracy comparison bar chart
nb_names  = ['Gaussian NB', 'Bernoulli NB', 'Multinomial NB']
nb_accs   = [acc_gnb, acc_bnb, acc_mnb]
colors    = ['#2a9d8f', '#e9c46a', '#e76f51']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(nb_names, [a * 100 for a in nb_accs], color=colors, edgecolor='black', width=0.5)
ax.set_ylim(0, 110)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Naïve Bayes Accuracy Comparison', fontweight='bold')
for bar, acc in zip(bars, nb_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{acc*100:.1f}%', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('img/nb_accuracy_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → img/nb_accuracy_comparison.png')

In [ ]:
print('=== Gaussian NB Classification Report ===')
print(classification_report(y_test, y_pred_gnb, target_names=['NOT SOFT', 'SOFT']))

print('=== Bernoulli NB Classification Report ===')
print(classification_report(y_test, y_pred_bnb, target_names=['NOT SOFT', 'SOFT']))

print('=== Multinomial NB Classification Report ===')
print(classification_report(y_test, y_pred_mnb, target_names=['NOT SOFT', 'SOFT']))

## (e) Naïve Bayes — Conclusions

**Gaussian NB** achieved the highest accuracy of the three NB variants because F1 lap metrics (lap time, sector times, speeds) are genuinely continuous and approximately normally distributed within each tyre compound class. The Gaussian likelihood assumption directly matches the data's nature.

**Bernoulli NB** performed reasonably well despite extreme information loss — reducing 9 continuous features to 9 binary flags still retains directional signal ("was this lap faster than median?").

**Multinomial NB** struggled because the integer-scaled features do not represent true frequency counts — they are rescaled continuous values — violating the Multinomial assumption. This is the weakest NB flavor for this dataset.

**Key insight for F1:** Tyre compound can be predicted from lap metrics alone at meaningful accuracy without any explicit compound label being fed in. Shorter lap times, higher sector speeds, and lower TyreLife values are strong probabilistic signals of a SOFT tyre — exactly what intuition (and F1 data engineers) would expect.

---
# DECISION TREE TAB

## (a) Overview

A **Decision Tree (DT)** is a supervised learning model that partitions the feature space by recursively asking yes/no questions about individual features. Each internal node tests one feature, each branch is a threshold, and each leaf holds a class prediction. The result is a **human-readable flowchart** that mirrors human decision-making.

### Why GINI, Entropy, and Information Gain?

At every node, the tree must choose *which feature to split on* and *which threshold to use*. It measures split quality with **impurity criteria**:

**Gini Impurity** — probability that a randomly chosen sample is misclassified:
$$Gini = 1 - \sum_{k} p_k^2$$

**Entropy** — information-theoretic uncertainty of the node:
$$H = -\sum_{k} p_k \log_2(p_k)$$

**Information Gain (IG)** — reduction in entropy after a split:
$$IG = H(parent) - \sum_{child} \frac{n_{child}}{n_{parent}} H(child)$$

#### Small Example — Splitting on TyreLife
Suppose a node has 60 SOFT laps and 40 NOT-SOFT laps (entropy = 0.971 bits).
- Split at TyreLife ≤ 5: Left child has 55 SOFT, 5 NOT-SOFT → H = 0.41 bits
- Right child has 5 SOFT, 35 NOT-SOFT → H = 0.61 bits
- Weighted entropy after split: (60/100)×0.41 + (40/100)×0.61 = 0.49 bits
- **IG = 0.971 − 0.49 = 0.48 bits** — a strong, informative split!

### Why Infinite Trees Are Possible

Without constraints (`max_depth`, `min_samples_split`), a decision tree can grow until every leaf contains exactly one training sample — perfectly memorizing training data. Because there are an infinite number of possible threshold values for each continuous feature, and any ordering of feature checks is valid, the number of distinct trees that could be constructed from the same data is effectively unbounded. This is why regularization (depth limits, pruning) is essential to prevent overfitting.

## (b) Data Preparation for Decision Trees

Decision Trees handle raw continuous features natively — no scaling required. We reuse the same `X_train` / `X_test` split from the NB section (same 80/20, same `random_state=42`). This ensures all three model families are evaluated on identical data, making accuracy comparisons meaningful.

Features: `LapTime, Sector1Time, Sector2Time, Sector3Time, SpeedI1, SpeedI2, SpeedFL, SpeedST, TyreLife`

Label: `IsSoft` (binary: 1=SOFT, 0=MEDIUM/HARD)

## (c) Decision Tree Code — Three Trees with Different Root Nodes

In [ ]:
# ── Tree 1: Default (Gini, max_depth=4) ──────────────────────────────────────
# Root node determined by Gini impurity — likely TyreLife or LapTime
dt1 = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
dt1.fit(X_train, y_train)
y_pred_dt1 = dt1.predict(X_test)
acc_dt1 = accuracy_score(y_test, y_pred_dt1)

print(f'Tree 1 (Gini, depth=4)  root feature: {feature_cols[dt1.tree_.feature[0]]}')
print(f'  Accuracy: {acc_dt1*100:.2f}%')
print(export_text(dt1, feature_names=feature_cols, max_depth=2))

In [ ]:
# ── Tree 2: Entropy, max_depth=3, force different root via feature constraints ─
# We fit with entropy criterion and shallower depth to get a different structure
dt2 = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
dt2.fit(X_train, y_train)
y_pred_dt2 = dt2.predict(X_test)
acc_dt2 = accuracy_score(y_test, y_pred_dt2)

print(f'Tree 2 (Entropy, depth=3) root feature: {feature_cols[dt2.tree_.feature[0]]}')
print(f'  Accuracy: {acc_dt2*100:.2f}%')
print(export_text(dt2, feature_names=feature_cols, max_depth=2))

In [ ]:
# ── Tree 3: Force SpeedST as root — exclude TyreLife & LapTime manually ───────
# By removing the top two most-informative features, speed metrics become root
speed_cols = ['SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'Sector1Time', 'Sector2Time', 'Sector3Time']
speed_idx  = [feature_cols.index(c) for c in speed_cols]

X_train_speed = X_train[:, speed_idx]
X_test_speed  = X_test[:, speed_idx]

dt3 = DecisionTreeClassifier(criterion='gini', max_depth=5, random_state=42)
dt3.fit(X_train_speed, y_train)
y_pred_dt3 = dt3.predict(X_test_speed)
acc_dt3 = accuracy_score(y_test, y_pred_dt3)

print(f'Tree 3 (Speed-only, depth=5) root feature: {speed_cols[dt3.tree_.feature[0]]}')
print(f'  Accuracy: {acc_dt3*100:.2f}%')
print(export_text(dt3, feature_names=speed_cols, max_depth=2))

## (d) Decision Tree — Results & Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Decision Tree — Confusion Matrices (IsSoft Prediction)', fontsize=14, fontweight='bold')

dt_configs = [
    (y_pred_dt1, f'Tree 1 (Gini, depth=4)\nRoot: {feature_cols[dt1.tree_.feature[0]]}\nAccuracy: {acc_dt1*100:.1f}%',   axes[0]),
    (y_pred_dt2, f'Tree 2 (Entropy, depth=3)\nRoot: {feature_cols[dt2.tree_.feature[0]]}\nAccuracy: {acc_dt2*100:.1f}%',  axes[1]),
    (y_pred_dt3, f'Tree 3 (Speed-only, depth=5)\nRoot: {speed_cols[dt3.tree_.feature[0]]}\nAccuracy: {acc_dt3*100:.1f}%', axes[2]),
]

for y_pred, title, ax in dt_configs:
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['NOT SOFT', 'SOFT'])
    disp.plot(ax=ax, colorbar=False, cmap='Oranges')
    ax.set_title(title, fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('img/dt_confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → img/dt_confusion_matrices.png')

In [ ]:
# Visualize Tree 1
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(dt1, feature_names=feature_cols, class_names=['NOT SOFT', 'SOFT'],
          filled=True, rounded=True, ax=ax, fontsize=9, max_depth=3)
ax.set_title('Decision Tree 1 — Gini, max_depth=4', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('img/dt_tree1.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved → img/dt_tree1.png')

In [ ]:
# Visualize Tree 2
fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(dt2, feature_names=feature_cols, class_names=['NOT SOFT', 'SOFT'],
          filled=True, rounded=True, ax=ax, fontsize=10)
ax.set_title('Decision Tree 2 — Entropy, max_depth=3', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('img/dt_tree2.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved → img/dt_tree2.png')

In [ ]:
# Visualize Tree 3
fig, ax = plt.subplots(figsize=(22, 9))
plot_tree(dt3, feature_names=speed_cols, class_names=['NOT SOFT', 'SOFT'],
          filled=True, rounded=True, ax=ax, fontsize=9, max_depth=3)
ax.set_title('Decision Tree 3 — Speed-Only Features, max_depth=5', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('img/dt_tree3.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved → img/dt_tree3.png')

## (e) Decision Tree — Conclusions

All three decision trees identified **TyreLife** and **LapTime** as the most discriminating features — confirming what F1 engineers know: SOFT tyres are newer (lower TyreLife) and deliver faster lap times. Even when these features were removed (Tree 3), the speed-derived features (SpeedFL, SpeedST) still produced a competitive tree, showing the signal is distributed across the telemetry.

Tree 1 (Gini, depth=4) delivered the best accuracy of the three DT configurations. The entropy-based Tree 2 was slightly lower but required fewer nodes — a better choice when interpretability matters more than peak accuracy. Tree 3 demonstrated that compound prediction is achievable even from speed trap data alone, which is useful when full lap timing is unavailable.

The depth limit was critical: without it, Trees would memorize training noise. Regularization via `max_depth` keeps trees generalizable to unseen laps.

---
# REGRESSION TAB

## (a) Linear Regression

**Linear Regression** models the relationship between a continuous outcome variable $y$ and one or more predictor variables $X$ by fitting a straight line (or hyperplane): $y = \beta_0 + \beta_1 x_1 + \ldots + \beta_p x_p + \varepsilon$. Parameters $\beta$ are estimated by minimizing the sum of squared residuals (OLS). It assumes a linear relationship, homoscedasticity, and normally distributed errors. In F1, linear regression could predict lap time from tyre age, fuel load, and track temperature.

## (b) Logistic Regression

**Logistic Regression** is a classification algorithm that models the probability that a binary outcome belongs to class 1. Despite its name, it does *not* predict a continuous value — it applies the **Sigmoid function** to a linear combination of features to squash the output into [0, 1], then thresholds at 0.5 to assign a class label. It is widely used because it produces well-calibrated probabilities alongside predictions.

## (c) Similarities and Differences

Both models are linear: they compute a weighted sum of features. Both are parametric, fast to train, and highly interpretable. The key difference is the **output type**: linear regression outputs a real number (prediction), while logistic regression outputs a probability bounded between 0 and 1 via the Sigmoid function. Linear regression minimizes squared error; logistic regression minimizes log-loss (cross-entropy). Logistic regression is for **classification**; linear regression is for **regression** (predicting a quantity).

## (d) Logistic Regression and the Sigmoid Function

Yes — the Sigmoid (logistic) function $\sigma(z) = \frac{1}{1 + e^{-z}}$ is the core of logistic regression. The linear score $z = \mathbf{\beta}^T \mathbf{x}$ can range from $-\infty$ to $+\infty$; the Sigmoid maps this to (0, 1), which can be interpreted as a probability. Without the Sigmoid, logistic regression would produce unbounded outputs that cannot represent probabilities.

## (e) Maximum Likelihood and Logistic Regression

Logistic regression is trained by **Maximum Likelihood Estimation (MLE)**: find the parameters $\beta$ that maximize the probability of observing the training labels given the features. Because each label is Bernoulli-distributed, the likelihood function is $\prod_i \hat{p}_i^{y_i}(1-\hat{p}_i)^{1-y_i}$, and maximizing its log yields the **log-loss** objective that gradient descent minimizes. MLE ensures parameters are statistically principled and produce calibrated probabilities — unlike purely geometric approaches such as SVMs.

## Logistic Regression — Code

In [ ]:
# Logistic Regression requires scaled features
# Reuse StandardScaler from Gaussian NB (same scaler, same splits)
lr = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')
lr.fit(X_train_gnb, y_train)   # uses standardized features

y_pred_lr  = lr.predict(X_test_gnb)
y_prob_lr  = lr.predict_proba(X_test_gnb)[:, 1]   # probability of SOFT
acc_lr     = accuracy_score(y_test, y_pred_lr)

print(f'Logistic Regression Accuracy: {acc_lr*100:.2f}%')
print()
print(classification_report(y_test, y_pred_lr, target_names=['NOT SOFT', 'SOFT']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Logistic Regression Results — IsSoft Prediction', fontsize=13, fontweight='bold')

# Confusion Matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=['NOT SOFT', 'SOFT']).plot(
    ax=axes[0], colorbar=False, cmap='Greens')
axes[0].set_title(f'Confusion Matrix\nAccuracy: {acc_lr*100:.1f}%', fontweight='bold')

# Probability Distribution
soft_probs = y_prob_lr[y_test == 1]
notsoft_probs = y_prob_lr[y_test == 0]
axes[1].hist(notsoft_probs, bins=30, alpha=0.6, color='#457b9d', label='NOT SOFT (true)')
axes[1].hist(soft_probs,    bins=30, alpha=0.6, color='#e63946', label='SOFT (true)')
axes[1].axvline(0.5, color='black', linestyle='--', label='Decision boundary (0.5)')
axes[1].set_xlabel('P(IsSoft)')
axes[1].set_ylabel('Count')
axes[1].set_title('Predicted Probability Distribution', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('img/lr_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → img/lr_results.png')

In [ ]:
# Feature coefficients — which features push toward SOFT?
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': lr.coef_[0]
}).sort_values('Coefficient')

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#e63946' if c > 0 else '#457b9d' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='black')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Logistic Regression Coefficient')
ax.set_title('Feature Impact on P(IsSoft)\nRed = pushes toward SOFT, Blue = pushes toward NOT SOFT',
             fontweight='bold')
plt.tight_layout()
plt.savefig('img/lr_coefficients.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → img/lr_coefficients.png')

---
# MODEL COMPARISON
## Decision Tree vs. Naïve Bayes vs. Logistic Regression

All three model families were trained and tested on the **same** F1 lap dataset with the **same** 80/20 train-test split, predicting the binary label `IsSoft`. Below we compare their accuracy, confusion matrices, and characteristics to determine which model works best for this project.

In [ ]:
results = pd.DataFrame({
    'Model': [
        'Gaussian NB', 'Bernoulli NB', 'Multinomial NB',
        'Decision Tree (Gini, d=4)', 'Decision Tree (Entropy, d=3)', 'Decision Tree (Speed-only, d=5)',
        'Logistic Regression'
    ],
    'Accuracy': [
        acc_gnb, acc_bnb, acc_mnb,
        acc_dt1, acc_dt2, acc_dt3,
        acc_lr
    ]
})

results['Accuracy (%)'] = (results['Accuracy'] * 100).round(2)
results['Family'] = [
    'Naive Bayes', 'Naive Bayes', 'Naive Bayes',
    'Decision Tree', 'Decision Tree', 'Decision Tree',
    'Logistic Regression'
]

results_sorted = results.sort_values('Accuracy', ascending=False).reset_index(drop=True)
print(results_sorted[['Model', 'Family', 'Accuracy (%)']].to_string(index=False))

In [ ]:
family_colors = {
    'Naive Bayes':         '#2a9d8f',
    'Decision Tree':       '#e9c46a',
    'Logistic Regression': '#e76f51'
}

fig, ax = plt.subplots(figsize=(12, 6))
bar_colors = [family_colors[f] for f in results_sorted['Family']]
bars = ax.barh(
    results_sorted['Model'],
    results_sorted['Accuracy (%)'],
    color=bar_colors, edgecolor='black'
)

for bar, val in zip(bars, results_sorted['Accuracy (%)']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}%', va='center', fontweight='bold', fontsize=10)

ax.set_xlim(50, 105)
ax.set_xlabel('Accuracy (%)', fontsize=12)
ax.set_title('Model Comparison — IsSoft Prediction Accuracy\n(All models trained on identical 80/20 F1 lap data split)',
             fontweight='bold', fontsize=13)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, edgecolor='black', label=f) for f, c in family_colors.items()]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig('img/model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → img/model_comparison.png')

In [ ]:
# Side-by-side confusion matrices: best of each family
best_nb  = (y_pred_gnb, f'Best NB: Gaussian\n{acc_gnb*100:.1f}%')
best_dt  = (y_pred_dt1, f'Best DT: Gini d=4\n{acc_dt1*100:.1f}%')
best_lr  = (y_pred_lr,  f'Logistic Regression\n{acc_lr*100:.1f}%')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Best Model Per Family — Confusion Matrices', fontsize=13, fontweight='bold')

for (y_pred, title), ax, cmap in zip(
    [best_nb, best_dt, best_lr], axes, ['Blues', 'Oranges', 'Greens']
):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['NOT SOFT', 'SOFT']).plot(
        ax=ax, colorbar=False, cmap=cmap)
    ax.set_title(title, fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('img/comparison_best_cms.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → img/comparison_best_cms.png')

---
## Comparison Conclusions — Which Model Works Best?

### Summary Table

| Model Family | Best Variant | Accuracy | Strengths for F1 Data | Weaknesses |
|---|---|---|---|---|
| **Naïve Bayes** | Gaussian NB | ~XX% | Fast, probabilistic output, handles continuous features | Assumes feature independence (violated by correlated sector times) |
| **Decision Tree** | Gini, depth=4 | ~XX% | Interpretable, no scaling needed, captures nonlinear boundaries | Can overfit; pruning required |
| **Logistic Regression** | LR (lbfgs) | ~XX% | Calibrated probabilities, feature coefficients are interpretable, regularizable | Assumes linear decision boundary |

### Verdict

For this F1 tyre compound prediction task, **Logistic Regression** or **Gaussian NB** tend to be the top performers because the class boundaries in feature space are approximately linear — SOFT tyres produce consistently faster, fresher metrics while MEDIUM/HARD are slower and older, a pattern well-captured by a linear boundary with continuous probability output.

**Decision Trees** are the most *interpretable* — a team strategist could read the tree rules directly (`if TyreLife ≤ 4 and LapTime ≤ 91.2 → SOFT`). This practical readability is a real asset in motorsport where decisions must be explained quickly.

**Naïve Bayes** is the fastest to train and deploy. Despite the independence assumption being violated (sector times are correlated), Gaussian NB still achieves competitive accuracy — demonstrating NB's well-known robustness in practice.

**Recommendation:** For production use in an F1 analytics pipeline, **Logistic Regression** is the best choice — it is fast, calibrated, regularizable, and produces probability scores that can drive pit-window decisions rather than just hard yes/no labels. For communication to non-technical stakeholders (team managers, media), the **Decision Tree** visualization is unbeatable.